In [12]:
import pandas as pd

In [13]:
# 1. Load both datasets
df_crop = pd.read_csv('../data/processed/focused_crop_data.csv')
df_weather = pd.read_csv('../data/raw/nasa_monthly_weather_2022_2025.csv')

In [21]:
print((df_crop["Production"] == 0).sum())

164


In [14]:
# Convert YearMonth to string just in case it loaded as an integer
df_weather['YearMonth'] = df_weather['YearMonth'].astype(str)

def get_season_months(crop_year, season):
    """Returns a list of 'YYYYMM' strings corresponding to the specific season and year"""
    # Extract the starting year (e.g., '2022' from '2022-23')
    start_year = int(crop_year.split('-')[0])
    next_year = start_year + 1
    
    if season.strip() == 'Kharif':
        # June to October of the start year
        return [f"{start_year}{str(m).zfill(2)}" for m in range(6, 11)]
    
    elif season.strip() == 'Rabi':
        # November/December of start year + Jan/Feb/March of next year
        months = [f"{start_year}11", f"{start_year}12"]
        months += [f"{next_year}{str(m).zfill(2)}" for m in range(1, 4)]
        return months
        
    elif season.strip() == 'Summer':
        # April to June of the next year
        return [f"{next_year}{str(m).zfill(2)}" for m in range(4, 7)]
    
    return []

# 2. Function to fetch the aggregated weather for a specific row
def fetch_weather_for_row(row):
    district = row['District']
    state = row['State']
    target_months = get_season_months(row['Crop_Year'], row['Season'])
    
    # Filter weather data for this specific district and the target months
    local_weather = df_weather[(df_weather['District'] == district) & 
                               (df_weather['State'] == state) & 
                               (df_weather['YearMonth'].isin(target_months))]
    
    if local_weather.empty:
        return pd.Series({'Total_Rainfall_mm': None, 'Avg_Temperature_C': None})
    
    # Calculate Total Rainfall and Average Temperature for the season
    total_rain = local_weather['Rainfall_mm'].sum()
    avg_temp = local_weather['Temperature_C'].mean()
    
    return pd.Series({'Total_Rainfall_mm': total_rain, 'Avg_Temperature_C': avg_temp})

print("Merging datasets...")

# 3. Apply the function to snap the weather data onto the crop data
weather_metrics = df_crop.apply(fetch_weather_for_row, axis=1)
df_final = pd.concat([df_crop, weather_metrics], axis=1)

# 4. Clean up any rows where weather couldn't be found (the failed geocodes)
df_final = df_final.dropna(subset=['Total_Rainfall_mm', 'Avg_Temperature_C'])

# 5. Save the Ultimate Master Dataset!
df_final.to_csv('../data/processed/master_yield_dataset.csv', index=False)

print("\n--- Merge Complete! ---")
print(f"Final rows ready for Machine Learning: {len(df_final)}")

Merging datasets...

--- Merge Complete! ---
Final rows ready for Machine Learning: 5301


In [20]:
print((df_final["Production"] == 0).sum())

160
